OpenQARP's variational algorithms (e.g. `QAOA`) accept any optimizer that exposes a Scipy-style `minimize(objective_function, initial_parameters, callback)` interface — they are not tied to `ScipyOptimizer`.

This notebook shows how to wrap an external optimizer — the genetic algorithm (GA) from the [pymoo](https://pymoo.org/) library — behind that interface and use it as a drop-in replacement. Please install `pymoo` before continuing.

In [ ]:
from scipy.optimize import OptimizeResult
from pymoo.optimize import minimize
from pymoo.core.problem import Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
import networkx as nx
import numpy as np

from qarp import config

First, let's create a toy graph over which we will optimize the Max Cut problem. We set `config.seed` once and reuse it everywhere randomness is involved (graph generation, the GA optimizer below), so the whole notebook is reproducible from a single seed.

In [ ]:
n_nodes = 4
n_edges = 2
config.seed = 1234

G = nx.gnm_random_graph(n_nodes, n_edges, seed=config.seed)
for u, v in G.edges:
    G[u][v]['weight'] = 2

In [ ]:
from qarp.graphs import Graph

graph = Graph(G)
graph.plot()

We now wrap pymoo's GA in a small adapter that follows the same `minimize(func, x0, callback)` API as `ScipyOptimizer`, so it can be passed directly to `QAOA`. The same `config.seed` is forwarded to pymoo so the GA run is reproducible too.

In [ ]:
class PymooOptimizer:
    def __init__(self, algorithm=None, seed=None, **kwargs):
        self.algorithm = algorithm if algorithm else GA(pop_size=100)
        self.seed = seed

    def minimize(self, func, x0, callback):
        class CustomProblem(Problem):
            def __init__(self):
                super().__init__(n_var=len(x0), n_obj=1, xl=0, xu=2*np.pi)

            def _evaluate(self, x, out, *args, **kwargs):
                out["F"] = np.array([func(xi) for xi in x])

        problem = CustomProblem()
        res = minimize(problem, self.algorithm, seed=self.seed)

        result = OptimizeResult()
        result.x = res.X
        result.fun = res.F[0]
        result.success = True
        result.message = "Optimization terminated successfully."
        return result

In [ ]:
optimizer = PymooOptimizer(seed=config.seed)

We specify to the variational quantum algorithm that the new implemented optimizer has to be used instead of the defaults.

The following code will use it

In [ ]:
from qarp.algorithms import QAOA

n_layers = 2
qaoa = QAOA(graph, n_layers=n_layers, verbose=True, 
            initial_parameters=[0]*2*n_layers,
            optimizer=optimizer).build()

fun, x = qaoa.run()

In [ ]:
print("Energy", fun)
print("Opt. parameters", x)

Now, we will run the default optimizer to check and compare the final results.

In [ ]:
n_layers = 2
qaoa = QAOA(graph, n_layers=n_layers, verbose=False, 
            initial_parameters=[0]*2*n_layers).build()

fun, x = qaoa.run()

In [ ]:
print("Energy", fun)
print("Opt. parameters", x)

Both optimization processes return the same energy. 